# Are change-point alarms worth their cost?

We calibrate a fixed-horizon CUSUM threshold using independent stable streams and evaluate it on disjoint deployment streams. A second threshold is calibrated on autocorrelated AR(1) null streams. The comparison isolates the economic cost of unnecessary retraining.

A conformal-style rank correction for the maximum statistic offers a marginal, fixed-horizon false-alarm statement only when the calibration maxima and a new null maximum are exchangeable. This is not an anytime-valid detector, and it does not guarantee performance under drift.


In [ ]:
import os
import matplotlib.pyplot as plt
from IPython.display import display
from horizon_detector_economics import MaintenanceDesign, run_maintenance_study, summarize_maintenance
SMOKE = os.getenv('MAINTENANCE_SMOKE','0') == '1'
REPEATS = int(os.getenv('MAINTENANCE_REPEATS','6'))
design = MaintenanceDesign(n_fit=80 if SMOKE else 160, n_cal=35 if SMOKE else 80,
    n_live=45 if SMOKE else 120, calibration_streams=19 if SMOKE else 99,
    delay_values=(0,) if SMOKE else (0,12),
    refit_every=10 if SMOKE else 15, refit_window=70 if SMOKE else 100,
    action_costs=(0.6,) if SMOKE else (0.2,0.6,1.0,1.4),
    refit_costs=(0.,0.2) if SMOKE else (0.,0.05,0.25))
calibration, models, economics = run_maintenance_study(design, evaluation_repeats=REPEATS,
    scenarios=('low_variance_signal',) if SMOKE else ('low_variance_signal','high_variance_signal'),
    regimes=('stable','ar1') if SMOKE else ('stable','gradual_drift','ar1','gradual_ar1'))
operational, decision = summarize_maintenance(models,economics)
display(calibration)
display(operational)
display(decision.head(16))


## False alarms and maintenance costs

The null calibration is disjoint from evaluation. A low cost per refit can favour frequent updates, while high maintenance costs can favour a frozen estimator. Report false alarms on stable and AR(1) streams separately: calibrating for IID data need not control alarms on autocorrelated streams.


In [ ]:
focus=decision.loc[(decision.scenario=='low_variance_signal')&(decision.regime=='ar1')&(decision.action_cost==0.6)&(decision.delay==0)]
fig,ax=plt.subplots(figsize=(8,4))
for policy,part in focus.groupby('policy'):
    ax.plot(part.refit_cost,part.mean_net_regret,marker='o',label=policy)
ax.set(xlabel='Cost per retraining event',ylabel='Mean net economic regret',title='Maintenance decisions under autocorrelation')
ax.legend()
fig.tight_layout()
plt.show()
assert (economics.net_regret >= -1e-9).all()


## Interpretation

The calibration target is the probability of one or more CUSUM alarms in the chosen finite horizon, not a perpetual operating-system false-alarm guarantee. The simulation has a fixed deployment horizon, a specified AR(1) mechanism, and known synthetic conditional means used exclusively for evaluation. Six independent evaluation streams are exploratory; more are needed for precise error-rate estimates. The economic oracle is an unattainable benchmark and no causal intervention effect is identified.
